1. Detectamos nuestros PDF

In [1]:
from pathlib import Path

# Carpeta que contiene nuestros documentos
carpeta_datos = Path("datos")

# Buscar los archivos PDF
archivos_pdf = sorted(carpeta_datos.glob("*.pdf"))

print(f"Encontramos {len(archivos_pdf)} documentos PDF:\n")

for archivo in archivos_pdf:
    print("-", archivo.name)

Encontramos 9 documentos PDF:

- ATAJO_Dip Allanamientos-digital-a 150.pdf
- BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
- Derechos de la persona imputada _ Unidad Especial Estratégica para la Implementación del Sistema Procesal Penal Acusatorio.pdf
- ley_26529_derechos_paciente.pdf
- PROTECCION DE LOS DATOS.pdf
- TEXTO ACTUALIZADO - Ley 23984 - CODIGO PROCESAL PENAL _ Argentina.gob.ar.pdf
- TEXTO ACTUALIZADO - Ley 27350 - INVESTIGACION MEDICA Y CIENTIFICA _ Argentina.gob.ar.pdf
- TEXTO ACTUALIZADO - Resolución 800_2021 - MINISTERIO DE SALUD _ Argentina.gob.ar.pdf
- TEXTO ORIGINAL - Resolución 275_2016 - MINISTERIO DE SEGURIDAD _ Argentina.gob.ar.pdf


1.1 Extraer el texto de los documentos PdfReader: abre y lee un PDF.

len(lector.pages): cuenta sus páginas.

extract_text(): extrae el texto de cada página.

len(texto): cuenta los caracteres que pudimos recuperar.

In [2]:
from pypdf import PdfReader

# Recorremos los 9 documentos PDF
for archivo in archivos_pdf:

    # Abrimos el documento
    lector = PdfReader(archivo)

    # Contamos las páginas
    cantidad_paginas = len(lector.pages)

    # Extraemos el texto de todas las páginas
    texto = ""

    for pagina in lector.pages:
        texto += pagina.extract_text() or ""

    # Mostramos los resultados
    print(f"Documento: {archivo.name}")
    print(f"Páginas: {cantidad_paginas}")
    print(f"Caracteres extraídos: {len(texto)}")
    print("-" * 50)

Documento: ATAJO_Dip Allanamientos-digital-a 150.pdf
Páginas: 4
Caracteres extraídos: 7013
--------------------------------------------------
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Páginas: 12
Caracteres extraídos: 32052
--------------------------------------------------
Documento: Derechos de la persona imputada _ Unidad Especial Estratégica para la Implementación del Sistema Procesal Penal Acusatorio.pdf
Páginas: 1
Caracteres extraídos: 1034
--------------------------------------------------
Documento: ley_26529_derechos_paciente.pdf
Páginas: 5
Caracteres extraídos: 16092
--------------------------------------------------
Documento: PROTECCION DE LOS DATOS.pdf
Páginas: 14
Caracteres extraídos: 41984
--------------------------------------------------
Documento: TEXTO ACTUALIZADO - Ley 23984 - CODIGO PROCESAL PENAL _ Argentina.gob.ar.pdf
Páginas: 166
Caracteres extraídos: 346329
--------------------------------------------------


In [3]:
from pypdf import PdfReader

for archivo in archivos_pdf:
    lector = PdfReader(archivo)

    texto = "".join(
        pagina.extract_text() or ""
        for pagina in lector.pages
    )

    estado = "OK" if texto.strip() else "SIN TEXTO"

    print(f"{estado} | {archivo.name} | {len(texto)} caracteres")

OK | ATAJO_Dip Allanamientos-digital-a 150.pdf | 7013 caracteres
OK | BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf | 32052 caracteres
OK | Derechos de la persona imputada _ Unidad Especial Estratégica para la Implementación del Sistema Procesal Penal Acusatorio.pdf | 1034 caracteres
OK | ley_26529_derechos_paciente.pdf | 16092 caracteres
OK | PROTECCION DE LOS DATOS.pdf | 41984 caracteres
OK | TEXTO ACTUALIZADO - Ley 23984 - CODIGO PROCESAL PENAL _ Argentina.gob.ar.pdf | 346329 caracteres
OK | TEXTO ACTUALIZADO - Ley 27350 - INVESTIGACION MEDICA Y CIENTIFICA _ Argentina.gob.ar.pdf | 10015 caracteres
OK | TEXTO ACTUALIZADO - Resolución 800_2021 - MINISTERIO DE SALUD _ Argentina.gob.ar.pdf | 54307 caracteres
OK | TEXTO ORIGINAL - Resolución 275_2016 - MINISTERIO DE SEGURIDAD _ Argentina.gob.ar.pdf | 59783 caracteres


Dividir los documentos en fragmentos (chunking)
Ahora vamos a dividir los textos en partes más pequeñas para que nuestro RAG pueda recuperar información específica.

In [4]:
from langchain_core.documents import Document
from pypdf import PdfReader

documentos = []

for archivo in archivos_pdf:
    lector = PdfReader(archivo)

    for numero_pagina, pagina in enumerate(lector.pages, start=1):
        texto = pagina.extract_text() or ""

        if texto.strip():
            documento = Document(
                page_content=texto,
                metadata={
                    "fuente": archivo.name,
                    "pagina": numero_pagina
                }
            )

            documentos.append(documento)

print(f"Se cargaron {len(documentos)} páginas con texto.")

Se cargaron 268 páginas con texto.


2. Dividir los textos

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

fragmentos = divisor.split_documents(documentos)

print(f"Se generaron {len(fragmentos)} fragmentos.")

NameError: name 'documentos' is not defined

En este código:
- chunk_size=1000: cada fragmento tendrá aproximadamente hasta 1.000 caracteres.
- chunk_overlap=200: compartirá 200 caracteres con el fragmento siguiente para evitar perder contexto al dividir el texto.
- split_documents(): divide las páginas conservando sus metadatos.

In [ ]:
print("FUENTE:", fragmentos[0].metadata["fuente"])
print("PÁGINA:", fragmentos[0].metadata["pagina"])

print("\nCONTENIDO:")
print(fragmentos[0].page_content)

FUENTE: ATAJO_Dip Allanamientos-digital-a 150.pdf
PÁGINA: 1

CONTENIDO:
PROGRAMA DE ACCESO 
COMUNITARIO A LA JUSTICIA
PROCURACIÓN GENERAL DE LA NACIÓN
REPÚBLICA ARGENTINA
FISCAL
MINISTERIO PÚBLICO
Programa de Acceso Comunitario a la Justicia
Tte. Gral. Juan Domingo Perón 667, Piso 1° - Ciudad Autónoma de Buenos Aires 
(54-11) 6089-9081 | atajo@mpf.gob.ar
AGENCIA TERRITORIAL DE ACCESO A LA JUSTICIA
El Programa de Acceso Comunitario a la Justicia del Ministerio Público Fiscal de la Nación 
fue creado con la ﬁnalidad de facilitar la participación de las personas en el sistema de 
administración de justicia.  
Desde el programa se coordina el trabajo de las Agencias Territoriales de Acceso 
Comunitario a la Justicia y se acciona para acercar a las ﬁscalías y a los organismos públicos 
nacionales, provinciales y municipales con las necesidades jurídicas de la comunidad. 
ALLANAMIENTOS


In [ ]:
print(f"Total de documentos PDF: {len(archivos_pdf)}")
print(f"Total de páginas con texto: {len(documentos)}")
print(f"Total de fragmentos: {len(fragmentos)}")

Total de documentos PDF: 9
Total de páginas con texto: 268
Total de fragmentos: 778


Ahora tenemos 778 fragmentos de legislación y material sobre derechos preparados para convertirlos en embeddings.

In [ ]:
import json

# Convertimos los fragmentos en datos que podemos guardar
datos_fragmentos = []

for fragmento in fragmentos:
    datos_fragmentos.append({
        "texto": fragmento.page_content,
        "metadata": fragmento.metadata
    })

# Guardamos los fragmentos en un archivo JSON
with open("fragmentos.json", "w", encoding="utf-8") as archivo:
    json.dump(datos_fragmentos, archivo, ensure_ascii=False, indent=2)

print(f"¡Se guardaron {len(datos_fragmentos)} fragmentos correctamente!")

¡Se guardaron 778 fragmentos correctamente!
